# Export Model NeuroCheck: .pth → .onnx (dari Google Drive)

Jalankan cell dari atas ke bawah, urut. Yang perlu kamu ubah: **Cell 2** (URL repo) dan **Cell 4** (path file `.pth` di Drive kamu).

Alur: clone repo → install dependencies → mount Google Drive → copy 2 file `.pth` dari Drive → export ke `.onnx` (otomatis tervalidasi) → download hasilnya / langsung upload ke Hugging Face Hub.

## 1. Clone repo GitHub kamu

In [ ]:
# GANTI URL di bawah kalau repo kamu beda
REPO_URL = "https://github.com/pekaeltempatkita-cell/Brainscan-.git"

!git clone "$REPO_URL" project
%cd project

## 2. Install dependencies (khusus buat export, ada torch)

In [ ]:
!pip install -q -r requirements-export.txt

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Cari & salin 2 file `.pth` dari Drive

Dulu dijalankan, cek dulu di mana persis file kamu di Drive. Jalankan cell ini buat nge-scan seluruh Drive kamu cari file `.pth` -- biar gampang nemu path-nya, gak perlu buka Drive manual satu-satu.

In [ ]:
import glob

found = glob.glob('/content/drive/MyDrive/**/*.pth', recursive=True)
if not found:
    print("Gak ketemu file .pth di Drive kamu. Cek lagi udah ke-upload ke Drive belum.")
else:
    print(f"Ketemu {len(found)} file .pth:\n")
    for f in found:
        print(f)

Dari hasil di atas, copy-paste PATH LENGKAP 2 file kamu ke cell di bawah (yang model precheck & yang model utama 10 kelas), lalu jalankan.

In [ ]:
import shutil, os

# GANTI 2 baris di bawah sesuai path hasil pencarian di cell sebelumnya
PRECHECK_PTH_DI_DRIVE = "/content/drive/MyDrive/GANTI/path/precheck_brain_gate_best.pth"
MAIN_PTH_DI_DRIVE      = "/content/drive/MyDrive/GANTI/path/hybrid_vit_efficientnet_brain_best.pth"

os.makedirs("outputs/checkpoints", exist_ok=True)

# Nama tujuan HARUS PERSIS ini -- sesuai yang dibaca scripts/export_to_onnx.py
shutil.copy(PRECHECK_PTH_DI_DRIVE, "outputs/checkpoints/precheck_brain_gate_best.pth")
shutil.copy(MAIN_PTH_DI_DRIVE, "outputs/checkpoints/hybrid_vit_efficientnet_brain_best.pth")

print("Berhasil disalin. Isi folder outputs/checkpoints/ sekarang:")
print(os.listdir("outputs/checkpoints"))

## 5. Jalankan export ke ONNX

Script ini otomatis: load checkpoint → convert ke ONNX → validasi struktur → bandingkan angka keluaran PyTorch vs ONNX (harus hampir identik).

In [ ]:
!python scripts/export_to_onnx.py

Kalau di atas keluar tulisan **"[OK] Output PyTorch dan ONNX cocok."** buat kedua model → berhasil, lanjut ke langkah 6.

## 6A. (Opsi 1) Simpan hasil balik lagi ke Google Drive kamu

In [ ]:
import shutil, os

# GANTI folder tujuan di Drive kalau mau
TUJUAN_DI_DRIVE = "/content/drive/MyDrive/NeuroCheck_ONNX"
os.makedirs(TUJUAN_DI_DRIVE, exist_ok=True)

for f in os.listdir("onnx_models"):
    shutil.copy(f"onnx_models/{f}", os.path.join(TUJUAN_DI_DRIVE, f))
    print(f"Tersimpan: {os.path.join(TUJUAN_DI_DRIVE, f)}")

## 6B. (Opsi 2 -- lebih praktis) Langsung upload ke Hugging Face Hub dari sini

Gak perlu download/simpen manual, langsung dari Colab. Sebelum jalanin cell ini:
1. Buka https://huggingface.co/settings/tokens → buat token baru (tipe "Write")
2. Edit `HF_REPO_ID` di cell bawah sesuai repo model kamu (buat dulu di https://huggingface.co/new kalau belum ada)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste token "Write" kamu di sini

In [ ]:
# GANTI sesuai repo Hugging Face kamu, contoh: "markus/neurocheck-onnx"
HF_REPO_ID = "namamu/neurocheck-onnx"

import re
with open("scripts/upload_to_hf.py") as f:
    content = f.read()
content = re.sub(r'HF_REPO_ID = ".*?"', f'HF_REPO_ID = "{HF_REPO_ID}"', content, count=1)
with open("scripts/upload_to_hf.py", "w") as f:
    f.write(content)

!python scripts/upload_to_hf.py

## 7. Cek ukuran file (buat mastiin muat di RAM 512MB backend kamu)

In [ ]:
import os
for f in os.listdir("onnx_models"):
    path = os.path.join("onnx_models", f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{f}: {size_mb:.1f} MB")

Kalau total ukuran (precheck + model utama) di atas ~150-200MB, kasih tau saya -- ada teknik **quantization** buat kompres ukurannya 2-4x lipat (akurasi turun sedikit, tapi jauh lebih ringan buat backend dengan RAM terbatas).